# SpliceBERT v2 — Does a frozen pre-mRNA LM encode a splice site's **partner**?

**One notebook. Real hg38 sequence, real GENCODE splice sites. No synthetic constructs.**

### Why v2 exists — v1 was fatally confounded
| flaw | evidence (measured on the real v1 data) |
|---|---|
| **Positional label leak** | `dataset.py` set `acceptor = left_flank`, a *constant*. The true acceptor sat at position **200 in 100.0%** of examples. |
| **Geometry oracle** | Both endpoints were pinned to the window edges → a model reading **zero nucleotides** scores **PMR 1.000**. |
| **Length channel** | `len(seq) = 400 + exon_len` exactly → length-only adversary **0.495**. |
| **Site detection suffices** | Decoys were *spurious* AG/GT motifs → an **oracle site detector scores PMR 1.000**. Partner binding was never tested. |
| **Readout can't bind** | `score = f(h_a) + g(h_d) + L(len)` — no joint term. |

### What v2 changes
1. **Decoys are REAL annotated splice sites** — donors that exist in the same gene but that this acceptor **never** pairs with in *any* isoform. Site detection is useless *by construction*.
2. **Fixed 1024nt window**, filled with **real genome** (never PAD — pad-count is itself a length channel) → length carries **0 bits**.
3. **Acceptor offset randomised** over a range **independent of exon length**.
4. **Candidates permuted** by a seed derived from the *sequence*, never the label.
5. **Bilinear readout** (can express binding), with the **additive readout as a falsification control that must fail**.

### The identified stratum
`adversarial` = examples where **both** the nearest-downstream rule **and** the length prior prefer a *decoy*. There: length prior **0.000**, nearest-downstream **0.000**, oracle site detector **= chance**.

**Any model above chance there must be using partner information.**

## 0 · Setup

In [ ]:
import os, sys, glob, json, time, subprocess
from collections import Counter

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    return r.returncode

sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
sh("pip -q install transformers pyfaidx")

from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/splicebert_v2"
os.makedirs(DRIVE, exist_ok=True)

# Code is version-controlled, not a blob: clone the pinned branch.
REPO = "https://github.com/cam-berger/gfm-circuit-discovery.git"
BRANCH = "splicebert-v2-partner-binding"
if not os.path.isdir("/content/gfm"):
    sh(f"git clone -q --depth 1 -b {BRANCH} {REPO} /content/gfm")
sh("git -C /content/gfm log --oneline -1")
if "/content/gfm/src" not in sys.path:
    sys.path.insert(0, "/content/gfm/src")

import numpy as np
import torch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

## 1 · Resources — genome, annotation, frozen SpliceBERT
`hg38.fa` is a ~3GB one-time download; it and the GTF cache on Drive.

In [ ]:
GTF = f"{DRIVE}/gencode.v50.basic.annotation.gtf.gz"
FASTA = f"{DRIVE}/hg38.fa"

# SpliceBERT already exists on Drive from the v1 run -- find it, do not re-upload 400MB.
CANDIDATES = [
    f"{DRIVE}/SpliceBERT.1024nt",
    "/content/drive/MyDrive/splicebert_run/raw_data/models/"
    "splicebert_official_archive/extracted/models/SpliceBERT.1024nt",
]
MODEL = next((c for c in CANDIDATES if os.path.isdir(c)), None)
if MODEL is None:
    hits = glob.glob("/content/drive/MyDrive/**/SpliceBERT.1024nt", recursive=True)
    MODEL = hits[0] if hits else None
if not (MODEL and os.path.isfile(f"{MODEL}/config.json")):
    raise FileNotFoundError(
        "SpliceBERT.1024nt not found. Looked in:\n  " + "\n  ".join(CANDIDATES)
        + "\n  /content/drive/MyDrive/**/SpliceBERT.1024nt\n"
        "Set MODEL by hand to the folder containing config.json."
    )
print("model :", MODEL)

if not os.path.exists(GTF):
    print("downloading GENCODE ...")
    sh("wget -q -O " + GTF + " https://ftp.ebi.ac.uk/pub/databases/gencode/"
       "Gencode_human/release_50/gencode.v50.basic.annotation.gtf.gz")
if not os.path.exists(FASTA):
    print("downloading hg38 (~3GB, one time) ...")
    sh("wget -q -O " + FASTA + ".gz https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz")
    sh("gunzip -f " + FASTA + ".gz")
print("genome:", FASTA, os.path.exists(FASTA))
print("gtf   :", GTF, os.path.exists(GTF))

## 2 · Build the v2 dataset
Cached to Drive — a Colab disconnect never costs you this.

In [ ]:
from gfmcd.splicing.dataset import PyfaidxGenome
from gfmcd.splicing.partner_data import (
    PartnerConfig, build_partner_dataset, split_partner_examples,
    save_partner_jsonl, load_partner_jsonl, LengthPrior, stratify,
)

CACHE = f"{DRIVE}/partner_dataset.jsonl"
cfg = PartnerConfig()   # 1024nt window, acceptor offset U[100,300], real-genome filler
cfg.validate()

if os.path.exists(CACHE):
    X = load_partner_jsonl(CACHE)
    print(f"loaded cache: {len(X):,}")
else:
    t0 = time.time()
    X = build_partner_dataset(GTF, PyfaidxGenome(FASTA), cfg)   # FULL GENOME
    save_partner_jsonl(X, CACHE)
    print(f"built {len(X):,} examples in {time.time()-t0:.0f}s -> {CACHE}")

S = split_partner_examples(X)
print({k: len(v) for k, v in S.items()}, "  (test = chr8,chr9 SEALED)")

## 3 · 🚨 LEAK GUARDRAILS — hard gate
v1 shipped because nobody ran these.

In [ ]:
pos = Counter(x.acceptor_pos for x in X)
seqlens = {len(x.sequence) for x in X}
tix = Counter(x.true_index for x in X)
share = max(pos.values()) / len(X)

checks = [
    ("acceptor position is not constant", len(pos) >= 50, f"{len(pos)} distinct (v1 had 1)"),
    ("no acceptor position dominates", share < 0.10, f"max share {share:.1%} (v1: 100%)"),
    ("window length carries 0 bits", len(seqlens) == 1, f"len(seq) in {seqlens}"),
    ("true index is not always 0", tix[0] / len(X) < 0.75, f"index-0 share {tix[0]/len(X):.1%}"),
    ("every example is a forced choice", all(len(x.candidates) >= 2 for x in X), "min 2 candidates"),
]
ok = True
for name, passed, detail in checks:
    ok = ok and passed
    print(f"{name:36s} {'PASS' if passed else 'FAIL':5s} {detail}")
if not ok:
    raise AssertionError("LEAK GUARDRAIL FAILED - do not proceed")
print("\nAll leak guardrails pass.")

## 4 · Strata
`adversarial` = both the nearest-downstream rule *and* the length prior prefer a decoy.

In [ ]:
prior = LengthPrior([x.true_exon_length for x in S["train"]])   # fit on TRAIN only
for x in X:
    x.metadata["stratum"] = stratify(x, prior)

print(Counter(x.metadata["stratum"] for x in X))
VAL = S["val"]
V_ADV = [x for x in VAL if x.metadata["stratum"] == "adversarial"]
print(f"val: {len(VAL):,}   val-adversarial: {len(V_ADV):,}")

## 5 · 🚨 RIVAL TABLE — the pre-registered shortcut controls
**The gate:** on `adversarial`, the oracle site detector must be *at* chance. Only then is the task identified.

In [ ]:
from gfmcd.splicing.rivals import (
    PositionOnlyRival, GeometryOracleRival, LengthPriorRival, NearestDownstreamRival,
    OracleSiteDetectorRival, CandidateIndexRival, top1, chance_level,
)

mu_d = float(np.mean([x.true_donor_pos for x in S["train"]]))
mu_end = float(np.mean([len(x.sequence) - x.true_donor_pos for x in S["train"]]))
RIVALS = [
    PositionOnlyRival(mu_d), GeometryOracleRival(mu_end), LengthPriorRival(prior),
    NearestDownstreamRival(), OracleSiteDetectorRival(), CandidateIndexRival(),
]

def rival_table(exs, title):
    ch = chance_level(exs)
    print(f"\n{title}  (n={len(exs):,}, chance={ch:.3f})")
    print(f"  {'rival':22s} {'shortcut':26s} {'acc':>6s}")
    for r in RIVALS:
        a = top1(r, exs)
        flag = "   <-- ABOVE CHANCE" if a > ch + 0.05 else ""
        print(f"  {r.name:22s} {r.shortcut:26s} {a:6.3f}{flag}")
    return ch

CH_ALL = rival_table(VAL, "ALL val")
CH_ADV = rival_table(V_ADV, "ADVERSARIAL val   <-- the identified stratum")

osd = top1(OracleSiteDetectorRival(), V_ADV)
if abs(osd - CH_ADV) > 1e-6:
    raise AssertionError("oracle site detector is NOT at chance - the dataset is broken")
print(f"\nGATE PASS: perfect site detection buys nothing ({osd:.3f} == chance {CH_ADV:.3f}).")
print(f"Any model above {CH_ADV:.3f} on this stratum MUST be using partner information.")

## 6 · Precompute frozen embeddings ⚡
The backbone is frozen and tiny → embed **once**, cache only the site positions. Probes then train on the cache in seconds.

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tok = AutoTokenizer.from_pretrained(MODEL)
bb = AutoModelForMaskedLM.from_pretrained(MODEL, output_hidden_states=True).to(DEV).eval()
for p in bb.parameters():
    p.requires_grad_(False)

HID = bb.config.hidden_size
NL = bb.config.num_hidden_layers
MAXC = max(len(x.candidates) for x in X)
print(f"SpliceBERT: {NL} layers x {bb.config.num_attention_heads} heads, hidden {HID}, max candidates {MAXC}")

EMB = f"{DRIVE}/emb_last_layer.npz"

@torch.no_grad()
def embed(exs, bs=64):
    A = np.zeros((len(exs), HID), dtype=np.float16)
    D = np.zeros((len(exs), MAXC, HID), dtype=np.float16)
    M = np.zeros((len(exs), MAXC), dtype=bool)
    for i in range(0, len(exs), bs):
        chunk = exs[i:i + bs]
        enc = tok([" ".join(x.sequence) for x in chunk], return_tensors="pt", padding=True).to(DEV)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(DEV == "cuda")):
            h = bb(**enc).hidden_states[-1].float()
        h = h.cpu().numpy()
        for j, x in enumerate(chunk):
            A[i + j] = h[j, x.acceptor_pos + 1]          # +1 for [CLS]
            for k, d in enumerate(x.candidates):
                D[i + j, k] = h[j, d + 1]
                M[i + j, k] = True
        if (i // bs) % 40 == 0:
            print(f"   {i}/{len(exs)}", end="\r")
    return A, D, M

if os.path.exists(EMB):
    z = dict(np.load(EMB))
    print("loaded embedding cache")
else:
    t0 = time.time()
    z = {}
    for name in ["train", "val"]:
        a, d, m = embed(S[name])
        z[f"{name}_a"], z[f"{name}_d"], z[f"{name}_m"] = a, d, m
        print(f"  {name}: {a.shape}")
    np.savez_compressed(EMB, **z)
    print(f"embedded in {time.time()-t0:.0f}s -> {EMB}")

## 7 · Arm 1 — partner binding
**Bilinear** (can bind) vs **Additive control** (cannot — *must* be at chance) vs **AcceptorQuery** (reads `h_a` + offset, never `h_d`).

In [ ]:
from gfmcd.splicing.partner_readout import (
    CandidateBatch, BilinearPartnerReadout, AdditiveDonorReadout,
    AcceptorQueryReadout, loss_fn, top1 as t1,
)

def make_batch(exs, a, d, m):
    off = np.zeros(m.shape, dtype=np.float32)
    lab = np.zeros(len(exs), dtype=np.int64)
    for i, x in enumerate(exs):
        for k, c in enumerate(x.candidates):
            off[i, k] = c - x.acceptor_pos
        lab[i] = x.true_index
    return CandidateBatch(
        torch.tensor(a, dtype=torch.float32), torch.tensor(d, dtype=torch.float32),
        torch.tensor(off), torch.tensor(m), torch.tensor(lab),
    )

TR = make_batch(S["train"], z["train_a"], z["train_d"], z["train_m"])
VA = make_batch(S["val"], z["val_a"], z["val_d"], z["val_m"])
ADV = torch.tensor([i for i, x in enumerate(VAL) if x.metadata["stratum"] == "adversarial"])

def subset(b, idx):
    return CandidateBatch(b.acceptor[idx], b.donor[idx], b.offset[idx], b.mask[idx], b.labels[idx])

def train_probe(cls, seed, epochs=30, bs=256, lr=1e-3):
    torch.manual_seed(seed)
    model = cls(HID).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    n = TR.labels.shape[0]
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            b = subset(TR, perm[i:i + bs]).to(DEV)
            opt.zero_grad()
            loss_fn(model(b), b).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        v = VA.to(DEV)
        acc_all = t1(model(v), v).item()
        av = subset(VA, ADV).to(DEV)
        acc_adv = t1(model(av), av).item()
    return model, acc_all, acc_adv

RES = {}
MODELS = {}
for label, cls in [
    ("bilinear (hypothesis)", BilinearPartnerReadout),
    ("ADDITIVE CONTROL", AdditiveDonorReadout),
    ("acceptor-query", AcceptorQueryReadout),
]:
    runs = [train_probe(cls, s) for s in range(3)]
    MODELS[label] = runs[0][0]
    a = [r[1] for r in runs]
    d = [r[2] for r in runs]
    RES[label] = (float(np.mean(a)), float(np.std(a)), float(np.mean(d)), float(np.std(d)))
    print(f"{label:22s} all={RES[label][0]:.3f}+-{RES[label][1]:.3f}   "
          f"ADVERSARIAL={RES[label][2]:.3f}+-{RES[label][3]:.3f}")

## 8 · Verdict

In [ ]:
print(f"{'model / rival':24s} {'ALL':>10s} {'ADVERSARIAL':>13s}")
print("-" * 50)
print(f"{'chance':24s} {CH_ALL:10.3f} {CH_ADV:13.3f}")
for r in RIVALS:
    print(f"{r.name:24s} {top1(r, VAL):10.3f} {top1(r, V_ADV):13.3f}")
print("-" * 50)
for k, v in RES.items():
    print(f"{k:24s} {v[0]:10.3f} {v[2]:13.3f}")

BIL = RES["bilinear (hypothesis)"][2]
ADD = RES["ADDITIVE CONTROL"][2]
PASSED = (ADD <= CH_ADV + 0.05) and (BIL > CH_ADV + 0.05)

print("\n" + "=" * 60)
if ADD > CH_ADV + 0.05:
    print("CONTROL FAILED. The additive readout beats chance on the adversarial")
    print("stratum, but it provably CANNOT express partner binding. A shortcut")
    print("remains. DO NOT report the bilinear number - find the leak first.")
elif BIL > CH_ADV + 0.05:
    print("PARTNER BINDING DETECTED.")
    print(f"  bilinear {BIL:.3f} > chance {CH_ADV:.3f}; additive control at chance ({ADD:.3f}).")
    print("  Site detection, position, length, distance and list order are all")
    print("  neutralised by construction -> only partner information remains.")
else:
    print("NULL RESULT (publishable).")
    print(f"  bilinear {BIL:.3f} ~ chance {CH_ADV:.3f}. SpliceBERT's frozen representation")
    print("  does NOT encode partner identity beyond site-hood, length and distance.")
    print("  v1's apparent 'partner selection' was a positional leak + site detection.")
print("=" * 60)

json.dump({
    "chance": {"all": CH_ALL, "adversarial": CH_ADV},
    "rivals": {r.name: {"all": top1(r, VAL), "adversarial": top1(r, V_ADV)} for r in RIVALS},
    "models": {k: {"all": v[0], "all_sd": v[1], "adversarial": v[2], "adversarial_sd": v[3]}
               for k, v in RES.items()},
    "n": {"all": len(VAL), "adversarial": len(V_ADV)},
    "gate_passed": PASSED,
}, open(f"{DRIVE}/arm1_results.json", "w"), indent=2)
print(f"\nsaved -> {DRIVE}/arm1_results.json")

## 9 · Arm 2 — causal (runs only if Arm 1 passed)
Does the acceptor's representation *read* its partner? Mask the true donor → the score should collapse. Mask a real *decoy* donor → it should not.

In [ ]:
@torch.no_grad()
def masked_accuracy(model, exs, which, bs=32):
    hits = 0
    for i in range(0, len(exs), bs):
        chunk = exs[i:i + bs]
        seqs = []
        for x in chunk:
            s = list(x.sequence)
            if which == "true":
                tgt = x.true_donor_pos
            elif which == "decoy":
                tgt = x.real_decoy_donor_pos[0]
            else:
                tgt = None
            if tgt is not None:
                for p in range(max(0, tgt - 2), min(len(s), tgt + 2)):
                    s[p] = "N"
            seqs.append(" ".join(s))
        enc = tok(seqs, return_tensors="pt", padding=True).to(DEV)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(DEV == "cuda")):
            h = bb(**enc).hidden_states[-1].float()
        C = max(len(x.candidates) for x in chunk)
        A = torch.zeros(len(chunk), HID, device=DEV)
        D = torch.zeros(len(chunk), C, HID, device=DEV)
        M = torch.zeros(len(chunk), C, dtype=torch.bool, device=DEV)
        O = torch.zeros(len(chunk), C, device=DEV)
        L = torch.zeros(len(chunk), dtype=torch.long, device=DEV)
        for j, x in enumerate(chunk):
            A[j] = h[j, x.acceptor_pos + 1]
            for k, c in enumerate(x.candidates):
                D[j, k] = h[j, c + 1]
                M[j, k] = True
                O[j, k] = c - x.acceptor_pos
            L[j] = x.true_index
        b = CandidateBatch(A, D, O, M, L)
        hits += int((model(b).argmax(-1) == L).sum().item())
    return hits / len(exs)

if not PASSED:
    print("Arm 1 did not pass its gate - Arm 2 skipped by design.")
else:
    m = MODELS["bilinear (hypothesis)"]
    hard = [x for x in V_ADV if x.real_decoy_donor_pos]
    clean = masked_accuracy(m, hard, "none")
    true_masked = masked_accuracy(m, hard, "true")
    decoy_masked = masked_accuracy(m, hard, "decoy")
    print(f"adversarial stratum (n={len(hard):,}), bilinear readout:")
    print(f"  clean              : {clean:.3f}")
    print(f"  TRUE donor masked  : {true_masked:.3f}   <- should COLLAPSE toward chance ({CH_ADV:.3f})")
    print(f"  DECOY donor masked : {decoy_masked:.3f}   <- should NOT collapse (specificity)")
    json.dump({"clean": clean, "true_masked": true_masked, "decoy_masked": decoy_masked,
               "chance": CH_ADV, "n": len(hard)},
              open(f"{DRIVE}/arm2_causal.json", "w"), indent=2)
    print(f"\nsaved -> {DRIVE}/arm2_causal.json")